# Notebook 1 — Data Cleansing & Exploratory Data Analysis
### Immersion Aluminium Holding Furnace (1120 kg/ch) — Anomaly Detection Pipeline

**Purpose:** Load raw sensor data, perform "Before" EDA, apply strict domain-knowledge
cleansing rules, perform "After" EDA, and export a clean dataset for Notebook 2.

**Domain cleansing rules applied here:**
1. **Time slicing** — keep only March 2025 → August 2025 (drop Jan/Feb 2025).
2. **Full-Zero Rule** — drop rows where *every* feature (`molten_temp`, `heater1`,
   `heater2`, `voltage_avr`, `current_avr`, `power_total`) is exactly 0 (furnace off / no signal).
3. **Partial-Zero Rule** — rows with *some* zero features are **kept**, since a single
   sensor reading 0 while others are active is itself a candidate anomaly signature
   (e.g. `power_total == 0` while `molten_temp` is still hot).


In [ ]:
# =========================================================
# 0. ENVIRONMENT SETUP
# =========================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
pd.set_option("display.max_columns", None)

# ---- Kaggle paths ----
INPUT_PATH = "/kaggle/input/datasets/aliciakyoumi/cia-model-anomaly/"
OUTPUT_PATH = "/kaggle/working/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Core feature set used across the whole pipeline
CORE_FEATURES = ["molten_temp", "heater1", "heater2", "voltage_avr", "current_avr", "power_total"]

print("Input path :", INPUT_PATH)
print("Output path:", OUTPUT_PATH)
print("Files found:", os.listdir(INPUT_PATH) if os.path.exists(INPUT_PATH) else "PATH NOT FOUND (check dataset attachment)")


In [ ]:
# =========================================================
# 1. LOAD RAW DATA
# =========================================================
import os
import pandas as pd

def find_target_file(input_path, target_filename="m15.xlsx"):
    """Locate a specific file in the dataset directory."""
    for root, _, files in os.walk(input_path):
        for f in files:
            # Spesifik mencari file dengan nama m15.xlsx
            if f.lower() == target_filename.lower():
                return os.path.join(root, f)
                
    # Akan error jika m15.xlsx tidak ditemukan
    raise FileNotFoundError(f"{target_filename} not found under {input_path}")

# Panggil fungsi untuk mencari m15.xlsx
raw_path = find_target_file(INPUT_PATH, "m15.xlsx")
print("Target data file loaded:", raw_path)

# Gunakan read_excel untuk membaca file .xlsx
df_raw = pd.read_excel(raw_path)

print("\nRaw shape:", df_raw.shape)
df_raw.head()

In [ ]:
# =========================================================
# 1a. TIMESTAMP PARSING
# =========================================================
# Attempt to auto-detect the timestamp column
TIME_COL_CANDIDATES = ["timestamp", "datetime", "date", "time", "Timestamp", "Datetime"]
time_col = next((c for c in TIME_COL_CANDIDATES if c in df_raw.columns), df_raw.columns[0])
print("Using time column:", time_col)

df_raw[time_col] = pd.to_datetime(df_raw[time_col], errors="coerce")
df_raw = df_raw.sort_values(time_col).reset_index(drop=True)
df_raw = df_raw.rename(columns={time_col: "timestamp"})

# Verify all CORE_FEATURES exist
missing_cols = [c for c in CORE_FEATURES if c not in df_raw.columns]
if missing_cols:
    print("WARNING — missing expected columns:", missing_cols)
else:
    print("All core features present:", CORE_FEATURES)

df_raw.info()


In [ ]:
# =========================================================
# 1b. Statistik Deskriptif
# =========================================================
stats = df_raw[CORE_FEATURES].describe().T[['count', 'mean', 'std', 'min', 'max']]

stats.round (4)

## 2. Initial EDA — **BEFORE** Cleansing

In [ ]:
# =========================================================
# 2a. Shape, dtypes, missing values
# =========================================================
print("Shape (raw):", df_raw.shape)
print("\nDtypes:\n", df_raw.dtypes)

missing_before = df_raw[CORE_FEATURES].isna().sum().to_frame("missing_count")
missing_before["missing_pct"] = (missing_before["missing_count"] / len(df_raw) * 100).round(2)
missing_before


In [ ]:
# =========================================================
# 2b. Missing value bar chart (BEFORE)
# =========================================================
fig = px.bar(missing_before.reset_index(), x="index", y="missing_pct",
             title="BEFORE Cleansing — Missing Value % per Feature",
             labels={"index": "Feature", "missing_pct": "% Missing"},
             color="missing_pct", color_continuous_scale="Reds")
fig.show()


In [ ]:
# =========================================================
# 2c. Correlation heatmap (BEFORE)
# =========================================================
plt.figure(figsize=(8, 6))
corr_before = df_raw[CORE_FEATURES].corr()
sns.heatmap(corr_before, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("BEFORE Cleansing — Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# 2d. Time-series plots — molten_temp, current_avr, power_total (BEFORE)
# =========================================================
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                     subplot_titles=["molten_temp (raw)", "current_avr (raw)", "power_total (raw)"])

fig.add_trace(go.Scatter(x=df_raw["timestamp"], y=df_raw["molten_temp"],
                          mode="lines", line=dict(color="firebrick", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_raw["timestamp"], y=df_raw["current_avr"],
                          mode="lines", line=dict(color="darkorange", width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=df_raw["timestamp"], y=df_raw["power_total"],
                          mode="lines", line=dict(color="steelblue", width=1)), row=3, col=1)

fig.update_layout(height=800, title_text="BEFORE Cleansing — Raw Time Series", showlegend=False)
fig.show()


In [ ]:
# =========================================================
# 2e. Boxplots — outlier inspection (BEFORE)
# =========================================================
fig, axes = plt.subplots(1, len(CORE_FEATURES), figsize=(20, 5))
for ax, col in zip(axes, CORE_FEATURES):
    sns.boxplot(y=df_raw[col], ax=ax, color="salmon")
    ax.set_title(col)
plt.suptitle("BEFORE Cleansing — Boxplots per Feature")
plt.tight_layout()
plt.show()


## 3. Domain-Knowledge Cleansing Rules

In [ ]:
# =========================================================
# 3a. RULE 1 — TIME SLICING (keep March 2025 -> August 2025 only)
# =========================================================
df_clean = df_raw.copy()

start_date = "2025-03-01"
end_date   = "2025-09-01"  # exclusive upper bound covers all of August

before_n = len(df_clean)
df_clean = df_clean[(df_clean["timestamp"] >= start_date) & (df_clean["timestamp"] < end_date)].reset_index(drop=True)
after_n = len(df_clean)

print(f"Time slicing: {before_n} -> {after_n} rows "
      f"({before_n - after_n} rows dropped, kept Mar-Aug 2025 only)")
print("New date range:", df_clean['timestamp'].min(), "to", df_clean['timestamp'].max())


In [ ]:
# =========================================================
# 3b. RULE 2 — FULL-ZERO RULE
# Drop rows where ALL core features are exactly 0 (furnace fully idle / dead signal)
# =========================================================
before_n = len(df_clean)

full_zero_mask = (df_clean[CORE_FEATURES] == 0).all(axis=1)
n_full_zero = full_zero_mask.sum()

df_clean = df_clean[~full_zero_mask].reset_index(drop=True)
after_n = len(df_clean)

print(f"Full-Zero Rule: dropped {n_full_zero} rows where ALL of {CORE_FEATURES} == 0")
print(f"Shape: {before_n} -> {after_n}")


In [ ]:
# =========================================================
# 3c. RULE 3 — PARTIAL-ZERO RULE (explicitly NOT dropping)
# We verify these rows are retained — they are meaningful for anomaly detection
# (e.g. power_total == 0 while molten_temp still has a value = possible sensor/relay fault)
# =========================================================
partial_zero_mask = (df_clean[CORE_FEATURES] == 0).any(axis=1) & ~(df_clean[CORE_FEATURES] == 0).all(axis=1)
n_partial_zero = partial_zero_mask.sum()

print(f"Partial-zero rows retained (NOT dropped): {n_partial_zero} "
      f"({n_partial_zero/len(df_clean)*100:.2f}% of cleaned data)")
print("\nExample partial-zero rows:")
df_clean[partial_zero_mask].head(5)


In [ ]:
# =========================================================
# 3d. Handle remaining missing values (light touch — do not fabricate physical readings)
# Forward-fill short gaps (sensor dropout), leave longer gaps as NaN for the model layer
# =========================================================
df_clean[CORE_FEATURES] = df_clean[CORE_FEATURES].ffill(limit=3)

remaining_na = df_clean[CORE_FEATURES].isna().sum()
print("Remaining NaNs after limited forward-fill:\n", remaining_na)


## 4. Post-Cleansing EDA — **AFTER**

In [ ]:
# =========================================================
# 4a. Shape comparison summary
# =========================================================
comparison = pd.DataFrame({
    "stage": ["Raw", "Cleaned"],
    "rows": [len(df_raw), len(df_clean)],
    "date_min": [df_raw["timestamp"].min(), df_clean["timestamp"].min()],
    "date_max": [df_raw["timestamp"].max(), df_clean["timestamp"].max()],
})
comparison


In [ ]:
# =========================================================
# 4b. Correlation heatmap (AFTER)
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(corr_before, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("BEFORE — Correlation")

corr_after = df_clean[CORE_FEATURES].corr()
sns.heatmap(corr_after, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("AFTER — Correlation")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# 4c. Time-series (AFTER) — molten_temp, current_avr, power_total
# =========================================================
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                     subplot_titles=["molten_temp (cleaned)", "current_avr (cleaned)", "power_total (cleaned)"])

fig.add_trace(go.Scatter(x=df_clean["timestamp"], y=df_clean["molten_temp"],
                          mode="lines", line=dict(color="firebrick", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_clean["timestamp"], y=df_clean["current_avr"],
                          mode="lines", line=dict(color="darkorange", width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=df_clean["timestamp"], y=df_clean["power_total"],
                          mode="lines", line=dict(color="steelblue", width=1)), row=3, col=1)

fig.update_layout(height=800, title_text="AFTER Cleansing — Time Series", showlegend=False)
fig.show()


In [ ]:
# =========================================================
# 4d. Boxplots — BEFORE vs AFTER, side by side
# =========================================================
fig, axes = plt.subplots(2, len(CORE_FEATURES), figsize=(24, 10))
for i, col in enumerate(CORE_FEATURES):
    sns.boxplot(y=df_raw[col], ax=axes[0, i], color="salmon")
    axes[0, i].set_title(f"BEFORE: {col}")
    sns.boxplot(y=df_clean[col], ax=axes[1, i], color="mediumseagreen")
    axes[1, i].set_title(f"AFTER: {col}")
plt.suptitle("Boxplot Comparison — BEFORE vs AFTER Cleansing", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# 4e. Row-count reduction bar chart
# =========================================================
fig = px.bar(comparison, x="stage", y="rows", color="stage",
             title="Row Count — Before vs After Cleansing",
             text="rows")
fig.show()


In [ ]:
# =========================================================
# 4f. Distribution & Skewness Comparison (Before vs After)
# =========================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Fokus pada fitur kontinu utama
features_to_plot = ["molten_temp", "current_avr", "power_total"]

fig, axes = plt.subplots(len(features_to_plot), 2, figsize=(16, 12))

for i, col in enumerate(features_to_plot):
    # BEFORE (Raw Data) - Warna Merah/Salmon
    sns.histplot(df_raw[col].dropna(), ax=axes[i, 0], color="salmon", bins=40, kde=True)
    axes[i, 0].set_title(f"BEFORE: {col} Distribution")
    axes[i, 0].set_ylabel("Frekuensi")
    
    # AFTER (Cleaned Data) - Warna Hijau
    sns.histplot(df_clean[col].dropna(), ax=axes[i, 1], color="mediumseagreen", bins=40, kde=True)
    axes[i, 1].set_title(f"AFTER: {col} Distribution")
    axes[i, 1].set_ylabel("Frekuensi")

plt.suptitle("Distribusi Data (Histogram) — BEFORE vs AFTER Cleansing", y=1.02, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# =========================================================
# Kesimpulan Skewness Otomatis
# =========================================================
print("="*60)
print("KESIMPULAN SKEWNESS (KEMIRINGAN DISTRIBUSI DATA)")
print("="*60)

for col in features_to_plot:
    skew_before = df_raw[col].skew()
    skew_after = df_clean[col].skew()
    
    print(f"\nFitur: {col.upper()}")
    print(f" - Skewness Sebelum Cleansing : {skew_before:.3f}")
    print(f" - Skewness Sesudah Cleansing : {skew_after:.3f}")
    
    # Logika interpretasi skewness
    if abs(skew_after) <= 0.5:
        interpretasi = "Cukup simetris (mendekati distribusi normal)."
    elif skew_after > 0.5:
        interpretasi = "Right-skewed (menceng ke kanan), data menumpuk di nilai rendah dengan ekor panjang ke nilai tinggi."
    else:
        interpretasi = "Left-skewed (menceng ke kiri), data menumpuk di nilai tinggi dengan ekor panjang ke nilai rendah."
        
    print(f" - Kesimpulan Akhir: Distribusi {col} sesudah cleansing bersifat {interpretasi}")


# =========================================================
# Tambahan: Perhitungan Kurtosis & Interpretasi
# =========================================================
print("\n" + "="*60)
print("KESIMPULAN KURTOSIS (TINGKAT KERUNCINGAN DISTRIBUSI DATA)")
print("="*60)

for col in features_to_plot:
    kurt_before = df_raw[col].kurtosis()
    kurt_after = df_clean[col].kurtosis()
    
    print(f"\nFitur: {col.upper()}")
    print(f" - Kurtosis Sebelum Cleansing : {kurt_before:.3f}")
    print(f" - Kurtosis Sesudah Cleansing : {kurt_after:.3f}")
    
    # Logika interpretasi kurtosis (excess kurtosis pandas default, normal = 0)
    if kurt_after > 1.0:
        interpretasi = "Leptokurtic (puncak sangat runcing dan ekor tebal/banyak outlier)."
    elif kurt_after < -1.0:
        interpretasi = "Platykurtic (puncak datar atau landai)."
    else:
        interpretasi = "Mesokurtic (mendekati distribusi normal standar)."
        
    print(f" - Kesimpulan Akhir: Distribusi {col} sesudah cleansing bersifat {interpretasi}")
print("\n" + "="*60)

In [ ]:
# =========================================================
# Perbandingan Statistik Deskriptif (Before vs After Cleansing)
# =========================================================

# Mengambil statistik deskriptif untuk data RAW (Sebelum) dan CLEAN (Sesudah)
# Kita ambil parameter count, mean, std, min, 50% (median), max
stats_before = df_raw[CORE_FEATURES].describe().T[['count', 'mean', 'std', 'min', 'max']]
stats_after = df_clean[CORE_FEATURES].describe().T[['count', 'mean', 'std', 'min', 'max']]

# Mengubah nama kolom agar jelas perbedaannya
stats_before = stats_before.rename(columns=lambda x: f"{x}_before")
stats_after = stats_after.rename(columns=lambda x: f"{x}_after")

# Digabung menjadi satu tabel dataframe perbandingan
comparison_stats = pd.concat([stats_before, stats_after], axis=1)

# Merapikan urutan kolom (misal: min_before, min_after, mean_before, mean_after, dst)
ordered_cols = []
for metric in ['count', 'min', 'max', 'mean', 'std']:
    ordered_cols.extend([f"{metric}_before", f"{metric}_after"])

comparison_stats = comparison_stats[ordered_cols]

print("Tabel Perbandingan Statistik Deskriptif (Before vs After Cleansing):")
comparison_stats.round(4)

## 5. Save Cleaned Dataset

In [ ]:
# =========================================================
# 5. EXPORT
# =========================================================
out_file = os.path.join(OUTPUT_PATH, "cleaned_data.csv")
df_clean.to_csv(out_file, index=False)
print(f"Saved cleaned dataset -> {out_file}")
print("Final shape:", df_clean.shape)
df_clean.head()
